In [8]:
import torch
from at2v.tokenizer import TagBPETokenizer
from at2v.anitag2vec import SetupConfig, AniTag2VecRunner, AniTag2Vec
import torch.nn.functional as F
from at2v.dloader import MergeSet

In [9]:
TOKENIZER_PATH = "../checkpoints/token_dataset_c7359727bcee4f8b_vocab_size_5000_freq_3.json"
CONFIG_PATH = "../checkpoints/setup_params_dec65b57a17b7033_c7359727bcee4f8b.json"
MODEL_PATH = "../checkpoints/anitag2vec_dec65b57a17b7033_c7359727bcee4f8b_i128_e50_s60203_b256_p1871744.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = SetupConfig.load_from_file(CONFIG_PATH)
print(cfg)
tagtok = TagBPETokenizer(vocab_size=cfg.HYPERP_TAGTOK_VOCAB_SIZE, min_frequency=cfg.HYPERP_TAGTOK_MIN_FREQ)
tagtok.load(TOKENIZER_PATH)

anitag2vec = AniTag2Vec(
    vocab_size=tagtok.vocab_size,
    max_len_cut=cfg.HYPERP_TAGTOK_MAX_TOKEN_CLAMP,
    d_model=cfg.HYPERP_TRANSFORMER_D_MODEL,
    n_heads=cfg.HYPERP_TRANSFORMER_N_HEADS,
    n_layers=cfg.HYPERP_TRANSFORMER_N_LAYERS,
    output_emb=cfg.HYPERP_OUTPUT_EMB,
)
anitag2vec.to(device)
anitag2vec.load_state_dict(torch.load(MODEL_PATH))
anitag2vec.eval()
runner = AniTag2VecRunner(tagtok, anitag2vec)

SetupConfig(TRAINING_TAKE_EXAMPLES=70000, TRAINING_BATCH_SIZE=256, TRAINING_PERM_LIMIT=8, TRAINING_SUBARRAY_COUNT=7, TRAINING_EPOCHS=50, HYPERP_TAGTOK_MAX_TOKEN_CLAMP=128, HYPERP_TAGTOK_VOCAB_SIZE=5000, HYPERP_TAGTOK_MIN_FREQ=3, HYPERP_TRANSFORMER_D_MODEL=128, HYPERP_TRANSFORMER_N_HEADS=8, HYPERP_TRANSFORMER_N_LAYERS=2, HYPERP_OUTPUT_EMB=128)


In [10]:
data = MergeSet.from_file(f"../data/output/merged_tags_v2.json")
runner = AniTag2VecRunner(tagtok, anitag2vec)

In [11]:
def compare(a: str, b: str):
    ax = runner.run_inference_human([a])
    bx = runner.run_inference_human([b])
    howmuch = ((F.normalize(ax) @ F.normalize(bx).T).item())
    print(f"{howmuch:.2f}: '{a}' vs '{b}'")

In [20]:
print("Permutation invariance:")
compare("#1girl #1boy", "#1boy #1girl")
compare("#nijigasaki #rina_tennoji", "#rina_tennoji #nijigasaki")

print("\nOppositeness and likeliness:")
compare("#nikke", "#blue_archive")
compare("#hayase_yuuka", "#blue_archive")
compare("#♀", "#girl")
compare("#♂", "#boy")
compare("#girl", "#boy")
compare("#♂", "#♀")
compare("#♂", "#unrelated")

print("\nSubsetness behavior on known and new combinations:")
compare("#A #B #C", "#C #B #A")
compare("#A #B #C", "#B #C #A")
compare("#1girl", "#1boy #1girl")
compare("#1boy", "#1boy #1girl")
compare("#d #a", "#a #b #c #d #e")
compare("#a #e", "#a #b #c #d #e")
compare("#a #e #c", "#a #b #c #d #e")

print("\nVery unlikely combinations:")
compare("#foo #bar", "#bar #foo")
compare("#foo #bar", " #foo")

Permutation invariance:
1.00: '#1girl #1boy' vs '#1boy #1girl'
1.00: '#nijigasaki #rina_tennoji' vs '#rina_tennoji #nijigasaki'

Oppositeness and likeliness:
0.35: '#nikke' vs '#blue_archive'
0.54: '#hayase_yuuka' vs '#blue_archive'
0.84: '#♀' vs '#girl'
0.82: '#♂' vs '#boy'
0.82: '#girl' vs '#boy'
1.00: '#♂' vs '#♀'
0.63: '#♂' vs '#unrelated'

Subsetness behavior on known and new combinations:
1.00: '#A #B #C' vs '#C #B #A'
1.00: '#A #B #C' vs '#B #C #A'
0.91: '#1girl' vs '#1boy #1girl'
0.88: '#1boy' vs '#1boy #1girl'
0.51: '#d #a' vs '#a #b #c #d #e'
0.52: '#a #e' vs '#a #b #c #d #e'
0.73: '#a #e #c' vs '#a #b #c #d #e'

Very unlikely combinations:
1.00: '#foo #bar' vs '#bar #foo'
0.88: '#foo #bar' vs ' #foo'


In [13]:
from tqdm import tqdm
query = ["bokutachi wa benkyou ga dekinai", "mafuyu"]
batch = 1000 # oneshot 1000 embeddings is relatively cheap (1GB GPU memory)
best = []
for start in tqdm(range(0, len(data.real_examples), batch), desc="Processing"):
    end = start + batch
    ranked = runner.rank_cosim(
        query,
        data.real_examples[start:end],
        best=True
    )
    if len(ranked) > 0:
        best.append(ranked[0])
best.sort(key=lambda x: -x[0])

Processing: 100%|██████████| 13/13 [00:06<00:00,  1.99it/s]


In [14]:
print("Closest to:", ", ".join(query))
for index, (cosim, tags) in enumerate(best[:3]):
    print(f"# {index + 1}, {cosim.item():.2}: {', '.join(sorted(tags))}")

Closest to: bokutachi wa benkyou ga dekinai, mafuyu
# 1, 0.83: arai kazuki, asumi kominami, bokutachi wa benkyou ga dekinai, doujinshi, japanese, mafuyu kirisu, maruarai
# 2, 0.8: arai kazuki, bokutachi wa benkyou ga dekinai, doujinshi, japanese, mafuyu kirisu, maruarai, nariyuki yuiga
# 3, 0.78: bakemonogatari, catgirl, chinese, kimono, screenshots, tsukihi araragi
